In [ ]:
import json
import boto3
import random
import pandas as pd
from io import StringIO
from datetime import datetime

# Function to properly parse date time
def parse_release_date(date_str):
    if len(date_str) == 4:
        return datetime.strptime(date_str, "%Y")
    elif len(date_str) == 7:
        return datetime.strptime(date_str, "%Y-%m")
    elif len(date_str) == 10:
        return datetime.strptime(date_str, "%Y-%m-%d")
    return None

# Function to extract the albums from the data
def album(data):
    album_list = []

    for item in data['items']:
        row = item['item']['album']

        album_id = row['id']
        album_name = row['name']
        album_release_date = row['release_date']
        album_total_tracks = row['total_tracks']
        album_url = row['external_urls']['spotify']
        album_artists = ", ".join(artist["name"] for artist in row["artists"])

        album_element = {
            'album_id': album_id,
            'name': album_name,
            'release_date': album_release_date,
            'total_tracks': album_total_tracks,
            'url': album_url,
            'artist_name':album_artists
        }

        album_list.append(album_element)
    
    return album_list


# Function to extract the artists from the data
def artist(data):
    artist_list = []
    for row in data['items']:
        artists = row['item']['artists']

        for artist in artists:
            artist_id = artist['id']
            artist_name = artist['name']
            artist_url = artist['href']

            artist_element = {
                'artist_id': artist_id,
                'artist_name': artist_name,
                'external_url': artist_url
            }

            artist_list.append(artist_element)

    return artist_list


# Function to extract songs from the data
def songs(data):
    song_list = []

    for row in data['items']:
        song = row['item']

        song_id = song['id']
        song_name = song['name']
        song_duration = song['duration_ms']
        song_url = song['external_urls']['spotify']
        song_added = row['added_at']
        album_id = song['album']['id']
        artist_id = ", ".join(
            artist['id'] for artist in song['artists']
        )

        song_element = {
            'song_id': song_id,
            'song_name': song_name,
            'duration_ms': song_duration,
            'url': song_url,
            'song_added': song_added,
            'album_id': album_id,
            'artist_id': artist_id,
            'song_popularity': random.randint(75, 95)
        }

        song_list.append(song_element)

    return song_list



def lambda_handler(event, context):
    s3=boto3.client('s3')
    Bucket='spotify-etl-project-abhishek-anant'
    Key='raw_data/to_be_processed/'
    
    spotify_data=[]
    spotify_keys=[]

    for file in s3.list_objects(Bucket=Bucket, Prefix=Key)['Contents']:
        file_key = file['Key']
        if file_key.split('.')[-1] == 'json':
            response = s3.get_object(Bucket=Bucket, Key=file_key)
            content=response['Body']
            jsonObject=json.loads(content.read())
            spotify_data.append(jsonObject)
            spotify_keys.append(file_key)

    for data in spotify_data:
        album_list=album(data)
        artist_list=artist(data)
        song_list=songs(data)

        album_df=pd.DataFrame.from_dict(album_list)
        album_df=album_df.drop_duplicates(subset=['album_id'])

        artist_df=pd.DataFrame.from_dict(artist_list)
        artist_df=artist_df.drop_duplicates(subset=['artist_id'])

        song_df=pd.DataFrame.from_dict(song_list)
        song_df=song_df.drop_duplicates(subset=['song_id'])



        album_df['release_date'] = pd.to_datetime(
            album_df['release_date'],
            format='mixed'
        )
        song_df['song_added'] = pd.to_datetime(
            song_df['song_added'],
            format='mixed'
        )

        album_key = 'transformed_data/album_data/album_transformed_' + str(datetime.now()) + '.csv'
        album_buffer = StringIO()
        album_df.to_csv(album_buffer)
        album_content=album_buffer.getvalue()
        s3.put_object(Bucket=Bucket, Key=album_key, Body=album_content)

        artist_key = 'transformed_data/artist_data/artist_transformed_' + str(datetime.now()) + '.csv'
        artist_buffer = StringIO()
        artist_df.to_csv(artist_buffer)
        artist_content=artist_buffer.getvalue()
        s3.put_object(Bucket=Bucket, Key=artist_key, Body=artist_content)

        songs_key = 'transformed_data/songs_data/songs_transformed_' + str(datetime.now()) + '.csv'
        song_buffer = StringIO()
        song_df.to_csv(song_buffer)
        song_content=song_buffer.getvalue()
        s3.put_object(Bucket=Bucket, Key=songs_key, Body=song_content)

    s3_resource=boto3.resource('s3')
    for key in spotify_keys:
        copy_source = {
            'Bucket': Bucket,
            'Key': key
        }
        s3_resource.meta.client.copy(copy_source, Bucket, 'raw_data/processed/' + key.split('/')[-1])
        s3_resource.Object(Bucket, key).delete()

